## Olist E-commerc: Data Exploration with DuckDB

## 1. Setup the notebook

In this notebook we rload five tables ot the e-comerce dataset into DuckDB and explore them with SQL before designing a data ware house

### what we will do
1. load the CSV files into DuckDB 
2. Understand each table: structure, grain, keys
3. Check the relationshiop (foreigh keys)
4. find data quality issues
5. Build a star schema

 tables `customers`, `orders`, `order_items`, `products`, `sellers`

In [7]:
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

In [8]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
WAREHOUSE_DIR = DATA_DIR / "warehouse"
csv = RAW_DIR / "olist_orders_dataset.csv" 

In [9]:
print(duckdb.sql(f"SELECT count(*) from '{csv}'"))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        99441 │
└──────────────┘



### DuckDB lets query CSV data with SQL. The database file persists locally so later notebooks canreuse the loades raw tables

In [10]:
DB_PATH = WAREHOUSE_DIR / "ecommerce.duckdb"
con = duckdb.connect(str(DB_PATH))


In [11]:
def q(sql_str: str) -> pd.DataFrame:
    """run a SQL queru in DuckDB and return an pandas Dataframe."""
    return con.sql(sql_str).df()

## 2. load the source tables into a raw schema
A Schema is a namespace that groups tables. We keep the source copies in rawe so that they remain distinguishable from cleaned warehouse tables we will create later

In [12]:
RAW_TABLES: dict[str, dict[str, str]] = {
    "customers": {"customer_zip_code_prefix" : "VARCHAR"}, # keep leading zeros
    "orders": {},
    "order_items": {},
    "products": {},
    "sellers": {},
}

### Define some helper functions

In [13]:
def csv_path(table: str) -> Path:
    return RAW_DIR / f"olist_{table}_dataset.csv"
def check_source_files(tables) -> None:
    missing = [csv_path(t) for t in tables if not csv_path(t).exists()]
    if missing:
        names = "\n".join(f" - {p}" for p in missing)
        raise FileNotFoundError(f"Missing source files:\n{names}")

#### Define schema name for source data

In [14]:
RAW_SCHEMA = "raw"

#### Load data into source tables

In [21]:
def load_raw_table(con: duckdb.DuckDBPyConnection, table: str, column_types: dict[str, str]) -> int:
    target = f'{RAW_SCHEMA}."{table}"'
    source = con.read_csv(str(csv_path(table)), header=True, dtype=column_types or None)
    con.execute(f"DROP TABLE IF EXISTS {target}")
    source.to_table(target)
    return con.execute(f"SELECT count(*) FROM {target}").fetchone()[0]

### Sanity check if we have all required source files

In [22]:
check_source_files(RAW_TABLES)

### Create **RAW_SCHEMA** if it does not exit

In [23]:
con.execute(f"CREATE SCHEMA IF NOT EXIsTS {RAW_SCHEMA}")

### Execute **load_raw_table** to load all source data

In [24]:

row_counts = {
    t: load_raw_table(con, t, types)
    for t, types in RAW_TABLES.items()
}

### Check count of rows inserted into each `raw`.`<table>`

In [25]:
pd.DataFrame(row_counts.items(), columns=["table_name", "row_count"])

,table_name,row_count
0,customers,99441
1,orders,99441
2,order_items,112650
3,products,32951
4,sellers,3095


## 3. Look at each table

for eveery table we ask: **what does one row represent?** this is the grain. getting the grain right is the most important step in dimensional modeling

In [29]:
for t in RAW_TABLES:
    print(f"---- {t} ----")
    display(q(f"DESCRIBE raw.{t}")[["column_name", "column_type", "null"]])
    display(q(f"SELECT * FROM raw.{t} LIMIT 5"))

---- customers ----


,column_name,column_type,null
0,customer_id,VARCHAR,YES
1,customer_unique_id,VARCHAR,YES
2,customer_zip_code_prefix,VARCHAR,YES
3,customer_city,VARCHAR,YES
4,customer_state,VARCHAR,YES


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


---- orders ----


,column_name,column_type,null
0,order_id,VARCHAR,YES
1,customer_id,VARCHAR,YES
2,order_status,VARCHAR,YES
3,order_purchase_timestamp,TIMESTAMP,YES
4,order_approved_at,TIMESTAMP,YES
5,order_delivered_carrier_date,TIMESTAMP,YES
6,order_delivered_customer_date,TIMESTAMP,YES
7,order_estimated_delivery_date,TIMESTAMP,YES


...

,column_name,column_type,null
0,product_id,VARCHAR,YES
1,product_category_name,VARCHAR,YES
2,product_name_lenght,BIGINT,YES
3,product_description_lenght,BIGINT,YES
4,product_photos_qty,BIGINT,YES
5,product_weight_g,BIGINT,YES
6,product_length_cm,BIGINT,YES
7,product_height_cm,BIGINT,YES
8,product_width_cm,BIGINT,YES


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
3,cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13


---- sellers ----


,column_name,column_type,null
0,seller_id,VARCHAR,YES
1,seller_zip_code_prefix,VARCHAR,YES
2,seller_city,VARCHAR,YES
3,seller_state,VARCHAR,YES


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
